# TP02 OOP and design patterns for CSV cleaning


## 0. Setup and inspect the input

The path search supports opening Jupyter from either the project root or `TP`.
For another location, set `TP_DIR` explicitly to the folder containing the CSV.

In [1]:
from abc import ABC, abstractmethod
from collections.abc import Iterable
from contextlib import redirect_stdout
from functools import wraps
from hashlib import sha256
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import sys

import numpy as np
import pandas as pd
from IPython.display import display


DATA_FILE = "sample_data (2).csv"
candidates = (Path.cwd(), Path.cwd() / "TP")
TP_DIR = next((p for p in candidates if (p / DATA_FILE).is_file()), None)
if TP_DIR is None:
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE!r}. Open this notebook from TP or its parent folder, "
        "or set TP_DIR to your data folder."
    )

TP_DIR = TP_DIR.resolve()
INPUT_PATH = TP_DIR / DATA_FILE
OUTPUT_PATH = TP_DIR / "outputs" / "tp02_oop_cleaned.csv"
INPUT_DIGEST = sha256(INPUT_PATH.read_bytes()).hexdigest()
MEASUREMENTS = ["age", "height_cm", "weight_kg", "score"]

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")
print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

Python 3.12.3 | pandas 3.0.6
Input: D:\ITC3_AMS_2025\I5_AMS\Advance_DS\TP\sample_data (2).csv
Output: D:\ITC3_AMS_2025\I5_AMS\Advance_DS\TP\outputs\tp02_oop_cleaned.csv


In [2]:
raw_df = pd.read_csv(INPUT_PATH)
original_df = raw_df.copy(deep=True)
display(raw_df)
display(pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "distinct_observed": raw_df.nunique(),
}))
print(f"Shape: {raw_df.shape}; exact duplicate rows: {raw_df.duplicated().sum()}")

assert raw_df.shape == (7, 7)
assert raw_df.isna().sum().to_dict() == {
    "id": 0, "name": 0, "age": 3, "height_cm": 1,
    "weight_kg": 1, "city": 0, "score": 2,
}

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


,dtype,missing,distinct_observed
id,int64,0,7
name,str,0,7
age,float64,3,4
height_cm,float64,1,6
weight_kg,float64,1,6
city,str,0,4
score,float64,2,5


Shape: (7, 7); exact duplicate rows: 0


There are seven missing entries, all in measurements. `id` is numeric but represents
identity rather than a quantity. The supplied file has no duplicates or messy text,
so separate small fixtures below make those behaviors visible.

## 1. Exercise 1 — Encapsulate CSV reading

`__init__` stores the file path and initializes the cached table. `read()` returns
the loaded DataFrame; `preview(n)` prints only its first `n` rows. The `_df` name
marks internal state by convention. Returning a copy keeps callers from accidentally
editing that cached table. File and parsing errors propagate with their original details.

In [3]:
class CSVReader:
    """Read a CSV and retain an internal copy for previewing."""

    def __init__(self, file_path: str | Path) -> None:
        self.file_path = Path(file_path)
        self._df: pd.DataFrame | None = None

    def read(self) -> pd.DataFrame:
        """Reload the CSV and return a separate DataFrame."""
        self._df = pd.read_csv(self.file_path)
        return self._df.copy(deep=True)

    def preview(self, n: int = 5) -> None:
        """Print the top n rows, loading the file first if necessary."""
        if isinstance(n, bool) or not isinstance(n, int):
            raise TypeError("n must be an integer.")
        if n < 0:
            raise ValueError("n must be non-negative.")
        if self._df is None:
            self.read()
        print(self._df.head(n).to_string(index=False))

In [4]:
reader = CSVReader(INPUT_PATH)
df = reader.read()
reader.preview(3)

pd.testing.assert_frame_equal(df, raw_df)
assert len(df) == 7  # Previewing does not shrink the stored data.

 id    name  age  height_cm  weight_kg        city  score
  1   Alice 29.0      165.0       68.0    New York   85.0
  2     Bob  NaN      172.0        NaN Los Angeles   90.0
  3 Charlie 35.0      168.0       72.0     Chicago    NaN


**Checkpoint:** the preview shows Alice, Bob, and Charlie. `reader` is an object;
`CSVReader` is its class. Python supplies `self` automatically when you call
`reader.preview(3)`.

## 2. Exercise 2 — Select a missing-value strategy

The abstract interface specifies `handle(df)`. Each concrete strategy returns a new
DataFrame, leaving its input unchanged. `DataCleaner` contains a strategy and delegates
to it: this is composition. The shared interface lets different implementations be
used interchangeably: this is polymorphism.

| Strategy | Rule | Explicit limitation |
|---|---|---|
| `DropMissing` | Drop rows containing any missing value | May discard substantial data |
| `FillMean` | Fill numeric gaps with each column's observed mean | Text and entirely missing columns remain missing |
| `FillMode` | Fill with the first mode returned by pandas | Entirely missing columns remain missing; ties use the first mode |

`FillMean` excludes `id` by default. Nullable integer measurements are converted to
floating point when filling, so a fractional mean can be represented. The numerical
examples assume real, finite measurements or missing values.

In [5]:
class MissingValueStrategy(ABC):
    """Contract for missing-value handlers that do not mutate their inputs."""

    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a cleaned DataFrame."""
        raise NotImplementedError


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Keep only complete rows, retaining their original index labels."""
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def __init__(self, exclude_columns: Iterable[str] = ("id",)) -> None:
        self.exclude_columns = frozenset(exclude_columns)

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy(deep=True)
        for column in result.select_dtypes(include="number").columns:
            if column in self.exclude_columns:
                continue
            values = result[column]
            if values.isna().any() and values.notna().any():
                # Float supports fractional means even for nullable Int64 input.
                values = values.astype("float64")
                result[column] = values.fillna(values.mean())
        return result


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy(deep=True)
        for column in result.columns:
            if result[column].isna().any():
                modes = result[column].mode(dropna=True)
                if not modes.empty:
                    result[column] = result[column].fillna(modes.iloc[0])
        return result

In [6]:
class DataCleaner:
    """Delegate cleaning to a replaceable MissingValueStrategy."""

    def __init__(self, strategy: MissingValueStrategy) -> None:
        self.strategy = strategy

    @property
    def strategy(self) -> MissingValueStrategy:
        return self._strategy

    @strategy.setter
    def strategy(self, strategy: MissingValueStrategy) -> None:
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("strategy must implement MissingValueStrategy.")
        self._strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return the result produced by the selected strategy."""
        return self.strategy.handle(df)

In [7]:
strategy_results = {}
comparison_rows = []

for strategy in (DropMissing(), FillMean(), FillMode()):
    name = type(strategy).__name__
    result = DataCleaner(strategy).clean(raw_df)
    strategy_results[name] = result
    comparison_rows.append({
        "strategy": name,
        "rows_kept": len(result),
        "rows_removed": len(raw_df) - len(result),
        "missing_remaining": int(result.isna().sum().sum()),
    })

display(pd.DataFrame(comparison_rows).set_index("strategy"))
display(pd.DataFrame({
    "name": raw_df["name"],
    "original_age": raw_df["age"],
    "mean_filled_age": strategy_results["FillMean"]["age"],
    "mode_filled_age": strategy_results["FillMode"]["age"],
}))

assert strategy_results["DropMissing"]["name"].tolist() == ["Alice", "Eva"]
np.testing.assert_allclose(raw_df[MEASUREMENTS].mean(), [30.25, 167, 70, 86])
assert strategy_results["FillMean"].loc[1, "age"] == 30.25
assert strategy_results["FillMode"].loc[1, "age"] == 27
for name in ("FillMean", "FillMode"):
    assert strategy_results[name].shape == (7, 7)
    assert strategy_results[name].isna().sum().sum() == 0
pd.testing.assert_frame_equal(raw_df, original_df)

,rows_kept,rows_removed,missing_remaining
strategy,,,
DropMissing,2,5,0
FillMean,7,0,0
FillMode,7,0,0


,name,original_age,mean_filled_age,mode_filled_age
0,Alice,29.0,29.00,29.0
1,Bob,NaN,30.25,27.0
2,Charlie,35.0,35.00,35.0
3,David,NaN,30.25,27.0
4,Eva,27.0,27.00,27.0
5,Frank,30.0,30.00,30.0
6,Grace,NaN,30.25,27.0


In [8]:
# Switch behavior on the same object without editing DataCleaner.clean.
cleaner = DataCleaner(FillMean())
print("FillMean rows:", len(cleaner.clean(raw_df)))
cleaner.strategy = DropMissing()
print("DropMissing rows:", len(cleaner.clean(raw_df)))

FillMean rows: 7
DropMissing rows: 2


**Interpretation:** dropping incomplete rows keeps only 2 of 7 observations. Mean and
mode filling retain all seven, but make different assumptions. All observed ages occur
once, so they tie for the mode; pandas returns `27` first for these values. The absence
of missing values here is a property of this dataset, not a universal guarantee.

## 3. Exercise 3 — Add logging and timing decorators

`@wraps` preserves the wrapped method's name and documentation. Each wrapper forwards
arguments and returns the result. A failure is logged and re-raised; timing is printed
even if the method fails.

`LoggedDataCleaner` extends the earlier cleaner without redefining it. Its method uses
both required decorators. Python function decorators demonstrate reusable wrapping;
the class-based Decorator design pattern is a related but distinct implementation.

In [9]:
def log_action(func):
    """Print start and finish messages, or a failure message on exception."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}")
        try:
            result = func(*args, **kwargs)
        except Exception as error:
            print(f"Failed {func.__name__}: {type(error).__name__}")
            raise
        print(f"Finished {func.__name__}")
        return result
    return wrapper


def log_time(func):
    """Print elapsed wall-clock time while preserving returns and exceptions."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"{func.__name__} took {perf_counter() - start:.6f} seconds")
    return wrapper


class LoggedDataCleaner(DataCleaner):
    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean with the selected strategy and report execution details."""
        return super().clean(df)

In [10]:
logged_cleaner = LoggedDataCleaner(FillMean())
cleaned_df = logged_cleaner.clean(raw_df)
pd.testing.assert_frame_equal(cleaned_df, strategy_results["FillMean"])
assert LoggedDataCleaner.clean.__name__ == "clean"
display(cleaned_df)

Starting clean
clean took 0.001212 seconds
Finished clean


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


**Checkpoint:** messages appear as `Starting clean`, `clean took ... seconds`, then
`Finished clean`. The elapsed time varies. The stack is equivalent to
`log_action(log_time(clean))`, so the action wrapper runs outermost. Adding logging
through wrappers illustrates the Open/Closed Principle: extend behavior while keeping
the existing cleaning algorithm unchanged.

## 4. Exercise 4 — Create transformations through a factory

Every transformation exposes `apply(df)` and returns a separate table.
Min-max scaling uses $x'=(x-\min(x))/(\max(x)-\min(x))$.
Constant observed values map to zero; missing entries stay missing. Entirely missing
columns stay missing. Identifiers are excluded by default.

Text standardization preserves non-string values and missing entries. Deduplication
keeps the first complete row match and resets the index. The factory centralizes
construction using the keys `normalize`, `remove_duplicates`, and `standardize_text`.

In [11]:
class DataTransform(ABC):
    """Contract for transformations that leave their inputs unchanged."""

    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Min-max scale real numeric columns except explicitly excluded identifiers."""

    def __init__(self, exclude_columns: Iterable[str] = ("id",)) -> None:
        self.exclude_columns = frozenset(exclude_columns)

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy(deep=True)
        for column in result.select_dtypes(include="number").columns:
            if column in self.exclude_columns:
                continue
            values = result[column].astype("float64")
            if not values.notna().any():
                continue
            minimum, maximum = values.min(), values.max()
            span = maximum - minimum
            result[column] = (values - minimum) / (span if span != 0 else 1.0)
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates(keep="first").reset_index(drop=True).copy()


class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy(deep=True)
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return result

In [12]:
class TransformFactory:
    """Simple factory mapping a name to a new transformation object."""

    _registry: dict[str, type[DataTransform]] = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name: str, **kwargs) -> DataTransform:
        if not isinstance(name, str):
            raise TypeError("Transformation name must be a string.")
        key = name.strip().lower()
        if key not in cls._registry:
            choices = ", ".join(sorted(cls._registry))
            raise ValueError(f"Unknown transformation {name!r}. Choose from: {choices}.")
        return cls._registry[key](**kwargs)

In [13]:
text_df = TransformFactory.create("standardize_text").apply(cleaned_df)
scaled_df = TransformFactory.create("normalize").apply(text_df)
display(scaled_df.round(4))  # Round for display only, not for storage.

pd.testing.assert_series_equal(scaled_df["id"], raw_df["id"])
assert scaled_df.loc[0, "name"] == "alice"
assert scaled_df[MEASUREMENTS].apply(lambda col: col.between(0, 1).all()).all()
assert scaled_df.loc[0, "age"] == 0.25
assert scaled_df.loc[1, "age"] == 0.40625

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,0.2500,0.3333,0.4333,new york,0.5882
1,2,bob,0.4062,0.8000,0.5000,los angeles,0.8824
2,3,charlie,1.0000,0.5333,0.5667,chicago,0.6471
3,4,david,0.4062,0.4667,0.8333,houston,0.0000
4,5,eva,0.0000,0.0000,0.0000,new york,0.7647
5,6,frank,0.3750,1.0000,1.0000,los angeles,0.6471
6,7,grace,0.4062,0.1333,0.1667,chicago,1.0000


In [14]:
# This fixture makes the effect of transformation order observable.
messy_df = pd.DataFrame({"name": [" Dara ", "dara", "SOPHEA"], "age": [20, 20, 30]})
tidy_text = TransformFactory.create("standardize_text").apply(messy_df)
tidy_df = TransformFactory.create("remove_duplicates").apply(tidy_text)
display(tidy_df)

deduplicated_first = TransformFactory.create("remove_duplicates").apply(messy_df)
reverse_order = TransformFactory.create("standardize_text").apply(deduplicated_first)
assert len(tidy_df) == 2
assert len(reverse_order) == 3
assert reverse_order.duplicated().sum() == 1
print("Text then duplicates:", len(tidy_df), "rows")
print("Duplicates then text:", len(reverse_order), "rows")

,name,age
0,dara,20
1,sophea,30


Text then duplicates: 2 rows
Duplicates then text: 3 rows


**Interpretation:** trim and lowercase first to expose duplicates hidden by text
formatting. Duplicate removal compares every column, so two otherwise equal records
with different IDs remain distinct. This is a simple factory, not the inheritance-based
Factory Method pattern. To add a transformation, implement `DataTransform` and add a
factory entry; clients continue to request objects by name.

## 5. Exercise 5 — Assemble the Template Method pipeline

`DataPipeline.run()` defines the fixed order: **load → clean → transform → save**.
`CSVDataPipeline` implements the steps and inherits `run()` unchanged. It composes
the reader, decorated cleaner, and factory-created transformations from earlier exercises.

The pipeline rejects the same resolved input/output path, creates the output directory,
and saves without a pandas index column. Calling a processing step before loading raises
a clear error. Each `run()` reloads the input, so rerunning does not repeatedly scale an
already transformed table.

In [15]:
class DataPipeline(ABC):
    """Template Method: subclasses implement steps, while run defines their order."""

    def run(self) -> pd.DataFrame:
        self.load()
        self.clean()
        self.transform()
        return self.save()

    @abstractmethod
    def load(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def clean(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def transform(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def save(self) -> pd.DataFrame:
        """Persist and return a separate copy of the final data."""
        raise NotImplementedError

In [16]:
class CSVDataPipeline(DataPipeline):
    """CSV cleaning workflow assembled from reusable components."""

    def __init__(
        self,
        input_path: str | Path,
        output_path: str | Path,
        strategy: MissingValueStrategy,
        transform_names: Iterable[str] = (),
    ) -> None:
        input_path = Path(input_path).resolve()
        self.output_path = Path(output_path).resolve()
        if input_path == self.output_path:
            raise ValueError("Input and output paths must be different.")
        if isinstance(transform_names, str):
            raise TypeError("Use a sequence of names, such as ['normalize'].")
        self.reader = CSVReader(input_path)
        self.cleaner = LoggedDataCleaner(strategy)
        self.transforms = [TransformFactory.create(name) for name in transform_names]
        self.df: pd.DataFrame | None = None

    def _require_data(self) -> pd.DataFrame:
        if self.df is None:
            raise RuntimeError("No data loaded. Call run() or load() first.")
        return self.df

    def load(self) -> None:
        self.df = self.reader.read()

    def clean(self) -> None:
        self.df = self.cleaner.clean(self._require_data())

    def transform(self) -> None:
        data = self._require_data()
        for transform in self.transforms:
            data = transform.apply(data)
        self.df = data

    def save(self) -> pd.DataFrame:
        data = self._require_data()
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(self.output_path, index=False)
        print(f"Saved {len(data)} rows to {self.output_path.name}")
        return data.copy(deep=True)

In [17]:
pipeline = CSVDataPipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    strategy=FillMean(),
    transform_names=["standardize_text", "remove_duplicates", "normalize"],
)
final_df = pipeline.run()
display(final_df.round(4))

Starting clean
clean took 0.001217 seconds
Finished clean
Saved 7 rows to tp02_oop_cleaned.csv


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,0.2500,0.3333,0.4333,new york,0.5882
1,2,bob,0.4062,0.8000,0.5000,los angeles,0.8824
2,3,charlie,1.0000,0.5333,0.5667,chicago,0.6471
3,4,david,0.4062,0.4667,0.8333,houston,0.0000
4,5,eva,0.0000,0.0000,0.0000,new york,0.7647
5,6,frank,0.3750,1.0000,1.0000,los angeles,0.6471
6,7,grace,0.4062,0.1333,0.1667,chicago,1.0000


In [18]:
# Verify values and the file round-trip, not just successful execution.
saved_df = pd.read_csv(OUTPUT_PATH)
assert saved_df.shape == (7, 7)
assert saved_df.columns.tolist() == raw_df.columns.tolist()
assert saved_df.isna().sum().sum() == 0
assert saved_df.duplicated().sum() == 0
pd.testing.assert_series_equal(saved_df["id"], raw_df["id"])
assert saved_df["name"].tolist() == raw_df["name"].str.lower().tolist()
assert saved_df[MEASUREMENTS].apply(lambda col: col.between(0, 1).all()).all()
pd.testing.assert_frame_equal(saved_df, final_df, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(raw_df, original_df)
assert sha256(INPUT_PATH.read_bytes()).hexdigest() == INPUT_DIGEST

display(pd.DataFrame({
    "stage": ["original", "cleaned and transformed", "reloaded CSV"],
    "rows": [len(raw_df), len(final_df), len(saved_df)],
    "missing": [int(x.isna().sum().sum()) for x in (raw_df, final_df, saved_df)],
}))
print("CSV round-trip passed; original input bytes are unchanged.")

,stage,rows,missing
0,original,7,7
1,cleaned and transformed,7,0
2,reloaded CSV,7,0


CSV round-trip passed; original input bytes are unchanged.


## 6. Behavioral checks and edge cases

These checks cover behavior that the seven-row example cannot demonstrate: all-missing
columns, text gaps, fractional means for nullable integers, constant measurements,
unknown factory keys, decorator failures, premature pipeline steps, and reruns.
Expected exceptions are caught and checked, so **Run All** still completes successfully.
Temporary files are created in a disposable directory under `TP`.

In [19]:
def expect_error(error_type, action, message_fragment: str = "") -> None:
    """Require an operation to raise the expected error and optional message."""
    try:
        action()
    except error_type as error:
        assert message_fragment in str(error), str(error)
    else:
        raise AssertionError(f"Expected {error_type.__name__}.")


edge_df = pd.DataFrame({
    "id": pd.Series([1, pd.NA, 3], dtype="Int64"),
    "measurement": pd.Series([1, pd.NA, 4], dtype="Int64"),
    "constant": [5.0, np.nan, 5.0],
    "all_missing": [np.nan, np.nan, np.nan],
    "label": pd.Series([" A ", pd.NA, " A "], dtype="string"),
})
edge_original = edge_df.copy(deep=True)
edge_mean = FillMean().handle(edge_df)
assert edge_mean.loc[1, "measurement"] == 2.5
assert pd.isna(edge_mean.loc[1, "id"])
assert pd.isna(edge_mean.loc[1, "label"])
assert edge_mean["all_missing"].isna().all()

edge_mode = FillMode().handle(edge_df)
assert edge_mode.loc[1, "label"] == " A "
assert edge_mode.loc[1, "measurement"] == 1
assert edge_mode["all_missing"].isna().all()

edge_scaled = NormalizeColumns().apply(edge_df)
np.testing.assert_allclose(edge_scaled["measurement"], [0, np.nan, 1], equal_nan=True)
np.testing.assert_allclose(edge_scaled["constant"], [0, np.nan, 0], equal_nan=True)
assert edge_scaled["all_missing"].isna().all()
pd.testing.assert_series_equal(edge_scaled["id"], edge_df["id"])

edge_text = StandardizeText().apply(edge_df)
assert edge_text.loc[0, "label"] == "a"
assert pd.isna(edge_text.loc[1, "label"])
mixed = pd.DataFrame({"value": pd.Series([" B ", 42, None], dtype=object)})
mixed_result = StandardizeText().apply(mixed)
assert mixed_result.loc[0, "value"] == "b"
assert mixed_result.loc[1, "value"] == 42
assert pd.isna(mixed_result.loc[2, "value"])

for operation in (DropMissing(), FillMean(), FillMode()):
    assert operation.handle(edge_df.iloc[:0]).empty
for transform in (NormalizeColumns(), RemoveDuplicates(), StandardizeText()):
    assert transform.apply(edge_df.iloc[:0]).empty
pd.testing.assert_frame_equal(edge_df, edge_original)

expect_error(TypeError, lambda: MissingValueStrategy())
expect_error(TypeError, lambda: DataTransform())
expect_error(TypeError, lambda: DataPipeline())
expect_error(TypeError, lambda: DataCleaner("mean"), "MissingValueStrategy")
expect_error(ValueError, lambda: reader.preview(-1), "non-negative")
expect_error(TypeError, lambda: reader.preview(1.5), "integer")
expect_error(ValueError, lambda: TransformFactory.create("unknown"), "Choose from")
expect_error(TypeError, lambda: TransformFactory.create(None), "string")
expect_error(
    ValueError,
    lambda: CSVDataPipeline(INPUT_PATH, INPUT_PATH, FillMean()),
    "different",
)
assert isinstance(TransformFactory.create(" NORMALIZE "), NormalizeColumns)

# A caller's edits must not alter the CSVReader cache.
isolated_reader = CSVReader(INPUT_PATH)
returned_df = isolated_reader.read()
returned_df.loc[0, "name"] = "changed externally"
preview_output = StringIO()
with redirect_stdout(preview_output):
    isolated_reader.preview(1)
assert "Alice" in preview_output.getvalue()
assert "changed externally" not in preview_output.getvalue()

print("Strategy, transformation, interface, and reader checks passed.")

Strategy, transformation, interface, and reader checks passed.

In [20]:
# Check decorator returns, metadata, message order, and exception propagation.
success_output = StringIO()
with redirect_stdout(success_output):
    returned = logged_cleaner.clean(raw_df)
lines = success_output.getvalue().strip().splitlines()
assert lines[0] == "Starting clean"
assert lines[1].startswith("clean took ")
assert lines[2] == "Finished clean"
pd.testing.assert_frame_equal(returned, strategy_results["FillMean"])
assert LoggedDataCleaner.clean.__doc__ is not None


@log_action
@log_time
def deliberate_failure():
    raise ValueError("intentional test failure")


failure_output = StringIO()
with redirect_stdout(failure_output):
    expect_error(ValueError, deliberate_failure, "intentional test failure")
failure_log = failure_output.getvalue()
assert "Starting deliberate_failure" in failure_log
assert "deliberate_failure took " in failure_log
assert "Failed deliberate_failure: ValueError" in failure_log
assert "Finished deliberate_failure" not in failure_log
print("Decorator checks passed, including propagation of the original exception.")

Decorator checks passed, including propagation of the original exception.


In [21]:
# Test an independent CSV with messy text, a duplicate, and a missing measurement.
with TemporaryDirectory(prefix="tp02_checks_", dir=TP_DIR) as temp_dir:
    temp_dir = Path(temp_dir)
    fixture_path = temp_dir / "fixture.csv"
    fixture = pd.DataFrame({
        "id": [1, 1, 2],
        "name": [" Dara ", "dara", "SOPHEA"],
        "score": [10.0, 10.0, np.nan],
    })
    fixture.to_csv(fixture_path, index=False)
    fixture_bytes = fixture_path.read_bytes()
    test_pipeline = CSVDataPipeline(
        fixture_path,
        temp_dir / "nested" / "clean.csv",
        FillMean(),
        ["standardize_text", "remove_duplicates", "normalize"],
    )
    expect_error(RuntimeError, test_pipeline.clean, "No data loaded")
    expect_error(RuntimeError, test_pipeline.transform, "No data loaded")
    expect_error(RuntimeError, test_pipeline.save, "No data loaded")
    expect_error(FileNotFoundError, lambda: CSVReader(temp_dir / "absent.csv").read())

    with redirect_stdout(StringIO()):
        first_run = test_pipeline.run()
        second_run = test_pipeline.run()
    expected = pd.DataFrame({"id": [1, 2], "name": ["dara", "sophea"], "score": [0.0, 0.0]})
    pd.testing.assert_frame_equal(first_run, expected)
    pd.testing.assert_frame_equal(first_run, second_run)
    pd.testing.assert_frame_equal(pd.read_csv(test_pipeline.output_path), expected)
    assert fixture_path.read_bytes() == fixture_bytes

    # Swapping the pipeline's strategy changes row retention, not its implementation.
    drop_pipeline = CSVDataPipeline(
        INPUT_PATH, temp_dir / "drop.csv", DropMissing(), ["standardize_text"]
    )
    with redirect_stdout(StringIO()):
        dropped = drop_pipeline.run()
    assert dropped["name"].tolist() == ["alice", "eva"]

assert sha256(INPUT_PATH.read_bytes()).hexdigest() == INPUT_DIGEST
print("Integration checks passed: order, save/load, reruns, strategy swap, input preservation.")
print("All notebook checks passed.")

Integration checks passed: order, save/load, reruns, strategy swap, input preservation.
All notebook checks passed.


## 7. Design review and further practice

| Concept | Where it appears | Why it helps |
|---|---|---|
| Encapsulation | `CSVReader` stores its path and cached data | Reading behavior and state stay together |
| Abstraction | `MissingValueStrategy`, `DataTransform`, `DataPipeline` | Contracts describe what subclasses must implement |
| Inheritance | Concrete strategies and `CSVDataPipeline` | Implementations fulfill shared interfaces |
| Polymorphism | `strategy.handle(df)` and `transform.apply(df)` | Clients work with different implementations uniformly |
| Composition | Cleaner contains a strategy; pipeline contains components | Responsibilities remain small and replaceable |
| Decorators | `log_action` and `log_time` | Instrumentation can be reused across methods |
| Simple factory | `TransformFactory.create(name)` | Construction decisions have one location |
| Template Method | `DataPipeline.run()` | Workflow order is defined once |

**Data science judgment.** Mean imputation can reduce variance and distort relationships.
Mode imputation can overrepresent a frequent category; its mechanical use on identifiers
would usually be inappropriate. Dropping rows may bias a sample. Choose a rule using the
meaning of the variables and the missingness process, not only the number of remaining gaps.

These exercise components compute statistics from the table passed to them. For predictive
modeling, split training and evaluation data first, learn imputation/scaling statistics on
training data only, and reuse those fitted statistics on validation/test data. This notebook
is a batch-cleaning exercise, not a fitted machine-learning preprocessing estimator.

**Scope:** examples cover ordinary CSV data with real finite measurements, string/object
text columns, and unique column names. They do not define policies for infinities, complex
numbers, category-encoded text, mixed-type mode ties, or domain-specific outliers. Entirely
missing columns are left unresolved for a deliberate domain decision.

### Practice after the baseline passes

1. Implement `FillMedian` and pass it to `DataCleaner` without changing the cleaner.
2. Remove `normalize` from the pipeline to retain original measurement units.
3. Exclude another numeric identifier using
   `TransformFactory.create("normalize", exclude_columns=("id", "customer_id"))`.
4. Implement a column-name transformation and add its key to the factory.
5. Explain why text standardization must precede duplicate removal in the fixture.
6. Change the final strategy to `DropMissing()` and update the seven-row expectations.

**Completion:** all five required exercises are implemented, demonstrated, and checked.
The generated CSV is in `outputs/tp02_oop_cleaned.csv`. Restart and run all cells again
after making changes to verify that the notebook has no hidden execution dependencies.